# Class 13 Homework: Multiple Regression analysis

## 1. Overview

Today we revisited some concepts of linear regression from previous weeks. We focused on multiple regression and variations of linear regression that are useful for multidimensional data.

In this notebook, we will use and compare the different regression methods.
The main question is: in a big multivariate dataset with many features, **how do you decide which features you need to include in a regression?**

The notebook relies on the new ideas in Class 13 notebook on data preparation and cross validation, while building on the linear regression material in Classes 9 through 12.

We will analyse a dataset of marine snails, abalone (https://en.wikipedia.org/wiki/Abalone). The dataset contains measurements of physical features of abalone, along with their **age measured in terms of the number of rings on their shells**. The columns in the dataset are:

- **Sex**: This column indicates the sex of the abalone and is of object data type. There are three possible values: 'M' for male, 'F' for female, and 'I' for infant.

- **Length**: This column contains the longest shell measurement in mm and is of float (continuous) data type.

- **Diameter**: This column contains the measurement perpendicular to length in mm and is of float data type.

- **Height**: This column contains the height of the whole abalone in mm and is of float data type.

- **Whole weight**: This column contains the weight of the whole abalone in grams and is of float data type.

- **Shucked weight**: This column contains the weight of the meat of the abalone in grams and is of float data type.

- **Viscera weight**: This column contains the weight of the abalone's gut (after bleeding) in grams and is of float data type.

- **Shell weight**: This column contains the weight of the abalone's shell after being dried in grams and is of float data type.

- **Rings**: This column indicates the age of the abalone in terms of the number of rings on their shell, and is of integer data type.

We aim to use these measurements to predict the age of abalone without the need to perform the tedious task of counting the rings through a microscope. By analyzing the relationship between these physical features and the age of abalone, we can build a predictive model to estimate their age, which can be useful for various applications in aquaculture, ecology, and fisheries management.
![](https://images.unsplash.com/photo-1619968987472-4d1b2784592e?ixlib=rb-4.0.3&ixid=MnwxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8&auto=format&fit=crop&w=2070&q=80)

## 2. Load data

#### 2.1. Importing Dependencies

As usual, we start by importing Python modules we will use. This uses NumPy, pandas, seaborn, and selected functions from scikit-learn.

Throughout the notebook, if any functions are unclear, try searching for help on the modules and function.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

#### 2.2. Loading Data

**TASK:** Load the file ```abalone.csv``` located in the same directory as this notebook using ```read_csv()``` from pandas.

In [ ]:
df = ...

#### 2.3. Data summary

As usual, we start by having a look at the data. 

**TASK:** Use ```head()``` and ```info()``` on the dataframe to get a first idea.

## 3. Exploratory Data Analysis 

**TIP:** We have a lot to do in this notebook. Be efficient with the Exploratory Data analysis, and cross-check with the Class 10 notebook where we used the dataset to develop the idea of correlation heatmaps.

#### 3.1. Summary statistics

We start by looking at some summary statistics.

**TASK:** Use ```describe()``` on the dataframe to get some stats.

#### 3.1.2 Correlations

Correlations are useful tools for exploratory and predictive analysis. As we covered in week 5, if the correlation between two variables is $R$, then in a simple linear regression $R^2$ is the proportion of variance explained.

While we've computed correlations between two variables, in a multivariate analysis it's helpful to compute correlations between *all* pairs of variabls in a dataset. Try this now - class 10 homework goes into depth on this useful concept.

**TASK:** Let's look at the correlation between features using the ```corr()``` function on the dataframe.
Remember that you can only apply this to numeric columns, so you may need to check the help page for the `.corr` function or drop non-numeric columns.

**TASK:** Check out the summary statistics. Can you already spot some patterns? Remember, correlation tells us about *linear* statistical dependence. Some features are highly correlated - which ones are they? Is that surprising? What could this mean for our later regression?

#### 3.2. Visual data exploration

Before starting any data analysis, we should always first inspect the data by plotting it in different ways. 

**TASK:** As you use different ways of plotting the data, reflect on the information conveyed and which plots you find most useful and why.

#### 3.2.3. Pairplots

Pairplots can be useful to get an idea of the pairwise relations between variables.

**TASK:** Use ```pairplot()``` from ```seaborn``` to plot all variables in the dataframe against each other.

*Note: you could revise Class 10 notebook about sumamrizing these correlations with a heatmap. That will be useful in Class 14 and the group project.*

#### 3.2.1. Scatter plots

While the pairplot plots **every** pair of variables, it can also be useful to focus on specific pairs and to use other features to highlight data on a single scatter plot. Let's try that.

**TASK:** Use seaborn scatter plot to plot 'Length' vs 'Rings'. Don't forget to add a title and axis labels. Colour the dots by the 'Sex' feature. 

Then swap 'Length' and 'Sex' for other features. Do you see any interesting relationships?

#### 3.2.2. Violin plots

We've already seen that violin plots can be useful to visualise the distributions in different categories. Here, our only categorical variable is 'Sex'.

**TASK:** Make a violin plot of the Age ('Rings') distribution for each class of 'Sex'.

## 3.5. Preliminary findings from our exploratory data analysis

**TASK:** Reflect! What have we learned about our dataset so far? What does the correlation of the single features with age/Rings mean in terms of variation explained (think of $R^2$)?

- The distribution of abalone age varies across different sex categories, with females generally having a higher age than males and infants.
- Features are highly correlated amongst each other, indicating that they have a strong linear relationship (not so surprising given what they are).
- The age of abalones increases with their length, diameter, and height, suggesting that these physical measurements can be used to predict the age of abalones, but the correlation is weak.
- The pairplot shows that there are strong positive correlations between the physical measurements of abalones (length, diameter, height, and weight), but only weak correlations between these measurements and the number of rings. This suggests that the dependence may be nonlinear or that these physical measurements may not carry sufficient information for accurate prediction of age each by themselves. We will see if together they provide sufficient information to predict age or if additional information, such as weather patterns and location (hence food availability) would be required.


## 4. Model Preparation

As in the class 13 notebook, we have to prepare the data and model before we start with our regression.

#### 2.4. Data preparation

While in this course we have mostly been dealing with data stored as pandas data frames, the underlying mathematics of machine learning uses matrices of numbers, i.e. linear algebra. Although in Biology 2A we aren't going to go in to the details of the underlying mathematical methods, we still have to prepare the data by expressing it as only numbers. For now, treat these data preparation steps as a "recipe" to get the scikit-learn functions to run.

#### 2.4.1 Label encoding

We first convert the categorical values in "Sex" into numerical values. Again, this uses `LabelEncoder` from scikit-learn.

In [ ]:
le = LabelEncoder()
df['Sex'] = le.fit_transform(df['Sex'])

**TASK:** Use ```info()``` again to see how the label encoding has changed.

Fixing some data types for easier handling later on - some algorithms better deal with float (continuous) values rather than discrete integer values.

In [ ]:
df['Sex']=df['Sex'].astype('float')

**TASK:** Use ```info()``` yet again to see how the label encoding has changed.

#### 2.4.3. Prepare list of feature names

Because we are using all columns as features to predict `Rings`, we need a list of all the column names in ```df```, except `Rings`. In fact this needs to be a Python ```list```. So you need to first get the columns except Rings (the last one) and then turn it into a list.

**TASK:** Make a list of feature (column) names that we can use later on. 

In [ ]:
# There are at least 3 ways to do this: loc, listing/subsetting, typing out the names.
# Pick one and check it!

feature_columns = ...

In [ ]:
feature_columns

#### 4.1. Separate Features and Outcomes into matrices

**TASK:** Make array/matrix variables ```X``` for the feature values and ```y``` for the outcomes ('Rings'). Here we need to use ```.values``` to extract just the numbers, while losing the rest of the data frame structure.

(It might be helpful to note that 'Rings' is the last column, which can be accessed with the index ```-1```; or to use the `features` list that you made earlier)

In [ ]:
# one approach with loc
X = df.loc[:, feature_columns].values
y = # TRY THIS YOURSELF

In [ ]:
X

In [ ]:
y

#### 4.2. Split Data into Training and Testing

Again, we split our dataset into training and test data for cross-validation. Today, we learned why we do this and how important it is to see if our trained model generalises to previously unseen data, or if it overfits the training data, i.e. too closely follows random relations in it. 

**TASK:** Use the function ```train_test_split()``` on ```X, y``` to create subsets ```X_train, X_test, y_train, y_test```. Split the data 80:20 between training and test sets by fixing ```test_size``` and fix the ```random_state``` to any value of your choice.

*For the curious: what does that mean, fixing the random state, i.e. the seed? Try to find out...).*

In [ ]:
# edit this so it runs as described
# Note the syntax with multiple variables on the left of the "="
# This means that one function gives multiple outputs, and we track them all.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = EDITME, random_state = EDITME)

In [ ]:
# inspect your data!

In [ ]:
# try with different random state etc.

## 5. Ordinary linear regression

Now finally, we are ready to build a regressor for our abalone dataset. We start simple, using a multivariate ordinary linear regressor.

This is essentially the same as the ordinary least squares `ols` and linear regression that we calculated in weeks 5 and 6, with 2 differences. One difference is splitting data into training and test sets. Second diference is introducing the scikit-learn way of calculating this that generalizes to other regression models later in the notebook.

#### 5.1. Model fitting

**TASK:** Create a model object called ```linreg``` using ```LinearRegression()``` and fit it using ```fit()``` on the training data. 

In [ ]:
from sklearn.linear_model import LinearRegression

linreg = LinearRegression().fit(EDITME)

Familiarize yourself with the fitted model object by looking at the coefficients and the intercept that has been fit as well as other attributes.

In [ ]:
linreg.coef_

In [ ]:
linreg.intercept_

#### 5.2. Model test

Even if the fitted model performs well on the training data, it may not perform well on the test data. This happens when the model is overfitting random associations in the training data rather than detecting the true underlying trends that shape the relationship between features and outcomes. Let's check how the model performs on the test data. 

**TASK:** Use the ```predict()``` function on ```linreg``` to make predictions for the test set, storing the results in ```ypred```, and then use functions ```r2_score()``` and ```mean_absolute_error``` to check the model performance. 

In [ ]:
from sklearn.metrics import r2_score,mean_absolute_error

ypred = linreg.predict(EDITME)
r2_pred = r2_score(EDITME)
mae_pred = mean_absolute_error(y_test,ypred)
print('R^2 = {:.3f}\nMAE = {:.3f}'.format(r2_pred, mae_pred))

**Reflect:** Are we happy with the result? How does it compare to the variation in outcome explained by each single feature (correlation matrix)?

**TASK:** Make a scatter plot of the true vs the predicted ages. Don't forget to add informative title and axis labels.

In [ ]:
fig_testvspred = sns.scatterplot(...)

...

To evaluate the predictions visually, it is helpful to compare to a "perfect" prediction, i.e. the diagonal line or identity line "x = y".

Different plotting libraries have different ways of adding such lines. Seaborn is part of matplotlib, and you can use [`plt.axline`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.axline.html) to add straight lines. Remember that a diagonal line passes through `[0,0]` and has slope 1.

**TASK:** Add a diagonal line to your plot from above.

In [ ]:
# Edit this code to add the line to your plot above

fig_testvspred.axline(xy1 = ..., slope = ..., color = ...)


Note here how the diagonal line emphasises where the fit is good and where it is less good.

## 6. Comparison of different regressors

In the lecture today, we learned that there are many different regression methods, and the above linear regressor is the simplest one. The below material is more complicated. We will use variations of linear regression via *Lasso* and *Ridge* regularization to see if we can improve on the ordinary regressor or gain further insights on the importance of features. 

We will also include a *Random Forest* regressor for comparison.

If you feel curious, you can try out others from the `sklearn` library . The other options include `ElasticNet` and `GradientBoostingRegressor`, that are shown commented out in the code below.

Although we won't cover these in depth in this course, these are all important methods in modern machine learning, that are applied to data analysis in biology and other fields.

#### 6.1. Make a comparison pipeline

Below are some definitions that will help us automate the comparison of models by making "pipelines" that run models automatically for many different parameter values. They use the `make_pipeline` function to set up the regressors.

The algorithms that fit these have random components, so we can make them run reproducibly by giving a seed to a (pseudo)random number generator as we did earlier in the course. Here, the parameter `random_state` sets the seed.

At this point, we'll provide most of the code for you to run, check, and start to understand.

**TASK:** Go through the code and check what regressors we will compare. 

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge, Lasso 
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
# from sklearn.linear_model import ElasticNet
# from sklearn.ensemble import GradientBoostingRegressor

pipelines={
'lasso':make_pipeline(Lasso(random_state=1234)),
'ridge':make_pipeline(Ridge(random_state=1234)),
'rf':make_pipeline(RandomForestRegressor(random_state=1234)),


# 'enet':make_pipeline(ElasticNet(random_state=1234)),
# 'gb':make_pipeline(GradientBoostingRegressor(random_state=1234)),
}

We have learned that different methods each have their hyperparameters that can be adapted to affect the performance of the model. Below we specify values for these hyperparameters to be tried in the optimisation during cross-validation later on.

Remember, you can read the documentation to follow what is happening here.

**TASK:** Go through the code and sklearn function documentation and check what hyperparameters the different methods have. 

In [ ]:
hyperparagrid={
'lasso':{
    'lasso__alpha':[0.001,0.005,0.01,0.05,0.1,0.5,0.99]
},
'ridge':{
    'ridge__alpha':[0.001,0.005,0.01,0.05,0.1,0.5,0.99]
},
'rf':{
'randomforestregressor__min_samples_split':[2,4,6],
'randomforestregressor__min_samples_leaf':[1,2,3]
},
    
# 'gb':{
#     'gradientboostingregressor__alpha':[0.001,0.005,0.01,0.05,0.1,0.5,0.99]
# },

# 'enet':{
#     'elasticnet__alpha':[0.001,0.005,0.01,0.05,0.1,0.5,0.99]
# }

}

#### 6.1. Fit Models

We will use ```GridSearchCV()``` from the ```sklearn.model_selection``` library, which will perform cross-validation over the defined grid of hyperparameters to fit a model with optimized hyperparameters.

So that we can use the same code across multiple models, we will use a ```for``` loop to run these.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.exceptions import NotFittedError

**TASK:** 
1) Create an empty dictionary called ```fit_models```.
2) Make a ```for``` loop over ```algo,pipeline in pipelines.items()```
3) Within the loop, create a ```model``` object using ```GridSearchCV()``` on your current ```pipeline``` and ```hyperparagrid[algo]``` using 10-fold cross-validation.
4) Still within the loop, fit ```model``` using ```fit()``` on the training data and assign the fitted model to ```fit_models[algo]```

**Warning: this code will take roughly 1 minute to run. Be patient and wait for it.** As you're waiting for the code to run, try to understand what each of the steps does.

In [ ]:
fit_models = {}
for algo,pipeline in pipelines.items():
    model = GridSearchCV(pipeline,hyperparagrid[algo],cv=10,n_jobs=-1)
    try:
        print('Start training for {}'.format(algo))
        model.fit(X_train,y_train)
        fit_models[algo] = model
    except NotFittedError as e:
        print(repr(e))

Make sure to inspect this ```fit_models`` object once it's generated.

In [ ]:
fit_models

#### 6.2. Model assessment

As we've learned today, it's essential to test your model fitted on the training data on previously unseen test data. 

**TASK:** Compute and compare the $R^2$ and mean absolute errors of the different models on the test set. Which one performs best?

In [ ]:
for algo,model in fit_models.items():
    ypred = ADAPT FROM PREVIOUS CODE CHUNK USING X_test
    r2_pred = ...
    mae_pred = ...
    print('{}:\t R^2 = {:.3f}\nMAE = {:.3f}'.format(algo,r2_pred, mae_pred))

Now let's check for overfitting. For that we compare the $R^2$ computed on the training and test sets. 

**TASK:** Compute and compare the $R^2$ of the different models on the training set and compare them to the above $R^2$ on the test set. Is any of the models overfitting the training data?

In [ ]:
for algo,model in fit_models.items():
    ypred = model.predict(X_train)
    r2_pred = ... # remember to use y_train
    mae_pred = ... # remember to use y_train
    print('{}:\t R^2 = {:.3f}\nMAE = {:.3f}'.format(algo,r2_pred, mae_pred))

**TASK:** For each fitted model, plot the true observed values in the test set vs the predicted values just as you did in **§5.2**. Don't forget to add titles and axis labels, adding any information (e.g. $R^2$) that might be useful.

In [ ]:
# example of doing this one model at a time, starting with lasso
lasso_model = fit_models['lasso']
ypred = lasso_model.predict(X_test)

fig_testvspred = sns.scatterplot(x = y_test, y = ypred)

fig_testvspred.set_title("Lasso predictions ")
fig_testvspred.set_ylabel("True age (no. of rings)")
fig_testvspred.set_xlabel("Predicted age (no. of rings)")
fig_testvspred.axline(xy1 = [0,0], slope = 1, color = "red")


In [ ]:
# example of doing this in a for loop for all 3 models
# is given in the answers notebook. Try it yourself?

**Reflect:** Which of the models performs best? Which model would you trust most and why?

- Random forest performs best in terms of accuracy on the test data.
- But it also overfits as can be seen from the higher $R^2$ on the training data. 
- Given that the other models are simpler and achieve a similar $R^2$ without overfitting, they might be preferrable to the random forest regressor.

#### 6.3. Model analysis

The regression methods we used in this section can give us additional information to the ordinary linear regressor we used in the previous section. Let's look at what we can learn from the different models.

#### 6.3.1. The Lasso model

In the lecture today, we learned that Lasso regression can help us with feature selection, as it assigns 0 to coefficients of features that don't carry sufficient information (or that don't add information as they are linearly correlated to other features). This means that their values are ignored in the predictions of the fitted Lasso model. 

**TASK:** Make a bar plot of the coefficients. Try to order them by value using ```pd.Series(<coefficients>, features).sort_values(ascending=False)```. As always, annotate your plot with title and axis labels.

In [ ]:
lasso_model = ...
coefs_ordered = pd.Series(...).sort_values(ascending=False)

fig_coefs = sns.barplot(...)
## the next line of code rotates axis labels so that they are readable instead of printing over each other
fig_coefs.tick_params(axis='x', labelrotation=90)

**TASK:** Reflect what it means that the coefficients of Sex and Length are 0. We've seen earlier that Length is highly correlated with age/Rings - so why does it get assigned a 0 coefficient?

- Length is highly correlated with other features that were assigned non-zero coefficients. Because of the intercorrelation of coefficients, not all of them are needed to predict the outcome. Which one of highly correlated features gets zeroed out is more or less random. 

#### 6.3.2. The Ridge model

In the lecture today, we learned that Ridge regularization can help us with feature importance, as non-informative variables will have coefficients close to 0. 

**TASK:** Using ```plt.figure()``` make a bar plot of the coefficients. Try to order them by value using ```pd.Series(<coefficients>, features).sort_values(ascending=False)```. As always, annotate your plot with title and axis labels.

In [ ]:
ridge_model = ...
feature_importance = # analogous to Lasso analysis using .coef_, but for Ridge model


In [ ]:
## now make bar plot as for Lasso analysis

**TASK:** 
1) Reflect what the importance of different features tells us. How do the coefficients compare to those of the Lasso model.
2) To look at feature importance, we should always use scaled or normalised features, meaning that their values get rescaled so that they are all within the same range. Repeat the Ridge regression analysis (from **§6.1** up to here) using ```StandardScaler()``` in **§6.1**. How do the feature importances change?

#### 6.3.3. The Random Forest model

In the lecture, we haven't discussed Random Forest models in detail, but they are often a simple yet useful method for both regression and classification. The fitted model also comes with a list of feature importances. Let's look at those. 

**TASK:** You can access the feature importances via ```rf_model.best_estimator_.named_steps.randomforestregressor.feature_importances_```. Same as for the Ridge model, make a bar plot of the feature importances (ideally ordered) and annotate your plot. 

In [ ]:
rf_model = ...
feat_imp_rf = ...



In [ ]:
# make bar plot next

**TASK:** Reflect and compare the feature importances of the Random Forest model with those of the Ridge model. What are the differences, and which importances would you trust more. 

- Given that the RF model was overfitted, the feature importances should be taken with a scepticism. The Ridge feature importances are probably more trustable in this case. 

## 6. Conclusions

**TASK:** Look back over the whole notebook - what have we learned about the abalone data and the predictability of age (in terms of number of rings)?

- The physical measurements of abalones (such as length, diameter, and weight) can be used to estimate abalone age. 
- But we the accuracy is not very high.
- This is no complete surprise, since we already noticed in our exploratory data analysis that features are highly inter-correlated, whereas their individual correlations with Rings is rather weak.
- Nevertheless, we can use those features to predict age/rings keeping in mind that predictions are not highly accurate. 
- It may be required to consider additional information, such as weather patterns and location, in order to achieve the highest possible accuracy. Therefore, future research could focus on the inclusion of other, potentially environmental variables to estimate the age of abalones.

These kinds of approaches could be used in your group poster project, to ask which features are more or less important in predicting an interesting outcome in the dataset

This notebook has been adapted from: https://www.kaggle.com/code/anmolbajpai/abalone-age-prediction-regression